In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torchmetrics import Accuracy, F1Score
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [3]:
df = pd.read_csv('data/joined_with_tt_scores.csv')
df = df.drop(columns=['Unnamed: 0.1', 'Unnamed: 0'])

In [4]:
#average rating per user
user_avg_rating = df.groupby('userId')['rating'].mean()

#average rating for user per genre
genre_cols = ['(no genres listed)', 'Action', 'Adventure', 'Animation', 'Children', 'Comedy', 
              'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'IMAX', 
              'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']

# Step 1: Multiply each genre column by rating
weighted = df[genre_cols].multiply(df['rating'], axis=0)

# Step 2: Sum ratings per user per genre
genre_sum = weighted.groupby(df['userId']).sum()

# Step 3: Count how many ratings per genre per user
genre_count = df[genre_cols].groupby(df['userId']).sum()

# Step 4: Compute averages
genre_avg = genre_sum / genre_count

user_features = genre_avg.reset_index()

user_features = user_features.rename(
    columns={col: f"user_avg_{col}" for col in genre_cols}
)

In [5]:
# Movie average rating
movie_sum = df.groupby('movieId')['rating'].transform('sum')
movie_count = df.groupby('movieId')['rating'].transform('count')

df['movie_avg_loo'] = (movie_sum - df['rating']) / (movie_count - 1)

# Handle edge case (movies with only 1 rating)
global_avg = df['rating'].mean()
df['movie_avg_loo'] = df['movie_avg_loo'].fillna(global_avg)

In [6]:
#Merge dataframes
# Columns corresponding to the user features
user_cols = [col for col in user_features.columns if col.startswith('user_avg_')]

df = df.merge(user_features, on='userId', how='left')

# Compute global average of ratings
global_avg = df['rating'].mean()

# Add missing indicators
for col in user_cols:
    df[f"{col}_missing"] = df[col].isna().astype(int)

# Fill NaNs with global average
df[user_cols] = df[user_cols].fillna(global_avg)

# Fill NaNs in movie_avg_loo
df['movie_avg_loo'] = df['movie_avg_loo'].fillna(global_avg)

In [7]:
# Make sure genre booleans are numeric
global_avg = df['rating'].mean()
for g in genre_cols:
    df[f'user_avg_{g}'] = df[f'user_avg_{g}'].fillna(global_avg)
    df[g] = df[g].astype(int)
    df[f'interact_{g}'] = df[f'user_avg_{g}'] * df[g]

In [8]:
k = 5  # tunable constant
df['movie_avg_weighted'] = df['movie_avg_loo'] * (movie_count / (movie_count + k))

interaction_cols = [f'interact_{g}' for g in genre_cols]
feature_cols = ['two_tower_score', 'movie_avg_loo'] + interaction_cols

In [9]:
# Suppose feature_cols is a list of column names
df[feature_cols].to_csv("src/models/movie_features.csv", index=False)

In [10]:
# -------------------------------
# Prepare dataset
# -------------------------------
def transform_with_noise(X, scaler, noise_std=1e-3):
    noise = np.random.normal(0, noise_std, X.shape)
    return scaler.transform(X + noise)


X = df[feature_cols].values
y = df['rating'].values
mean_rating = y.mean()

# Scale numeric features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Map userId/movieId to 0-based indices
user2idx = {uid: i for i, uid in enumerate(df['userId'].unique())}
movie2idx = {mid: i for i, mid in enumerate(df['movieId'].unique())}

user_ids = df['userId'].map(user2idx).values
movie_ids = df['movieId'].map(movie2idx).values

n_users = len(user2idx)
n_movies = len(movie2idx)

# Split into train/test
X_train, X_test, y_train, y_test, user_train, user_test, movie_train, movie_test = train_test_split(
    X_scaled, y, user_ids, movie_ids, test_size=0.2, random_state=42
)

# Add tiny noise to training features
X_train_scaled = transform_with_noise(X_train, scaler)
X_test_scaled = X_test  # no noise at test

# Convert to tensors
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1,1)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).view(-1,1)

user_train_tensor = torch.tensor(user_train, dtype=torch.long)
user_test_tensor = torch.tensor(user_test, dtype=torch.long)
movie_train_tensor = torch.tensor(movie_train, dtype=torch.long)
movie_test_tensor = torch.tensor(movie_test, dtype=torch.long)

In [11]:
class MovieRecommenderEmb(nn.Module):
    def __init__(self, num_users, num_movies, embed_dim, input_dim, mean_rating, dropout=0.2):
        super().__init__()
        self.user_embed = nn.Embedding(num_users, embed_dim)
        self.movie_embed = nn.Embedding(num_movies, embed_dim)
        self.user_bias = nn.Embedding(num_users, 1)
        self.movie_bias = nn.Embedding(num_movies, 1)
        
        # Fully connected layers
        self.fc = nn.Sequential(
            nn.Linear(input_dim + 3*embed_dim, 64),  # note +3*embed_dim
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1)
        )
        nn.init.constant_(self.user_bias.weight, 0.0)
        nn.init.constant_(self.movie_bias.weight, 0.0)
        self.mean_rating = mean_rating

    def forward(self, numeric_x, user_x, movie_x):
        user_vec = self.user_embed(user_x) / np.sqrt(self.user_embed.embedding_dim)
        movie_vec = self.movie_embed(movie_x) / np.sqrt(self.movie_embed.embedding_dim)
        interaction = user_vec * movie_vec   # element-wise product
        
        x = torch.cat([numeric_x, user_vec, movie_vec, interaction], dim=1)
        base = self.fc(x)
        bias = self.user_bias(user_x) + self.movie_bias(movie_x)
        return base + bias + self.mean_rating

In [12]:
input_dim = len(feature_cols)
embed_dim = 32
n_users = len(user2idx)
n_movies = len(movie2idx)

model = MovieRecommenderEmb(n_users, n_movies, embed_dim, input_dim, mean_rating, dropout=0.2)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=5e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, min_lr=1e-5)

best_test_loss = float('inf')
patience = 10
counter = 0
epochs = 200

for epoch in range(epochs):
    model.train()
    preds = model(X_train_tensor, user_train_tensor, movie_train_tensor)
    loss = criterion(preds, y_train_tensor)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    model.eval()
    with torch.no_grad():
        test_preds = model(X_test_tensor, user_test_tensor, movie_test_tensor)
        test_loss = criterion(test_preds, y_test_tensor)
    
    scheduler.step(test_loss)
    
    if test_loss < best_test_loss - 1e-4:
        best_test_loss = test_loss
        best_model_state = model.state_dict()
        counter = 0
    else:
        counter += 1
    
    if (epoch+1) % 10 == 0:
        print(f"Epoch {epoch+1}, Train: {loss.item():.4f}, Test: {test_loss.item():.4f}")
    
    if counter >= patience:
        print(f"\nEarly stopping at epoch {epoch+1}")
        break

# Restore best model
model.load_state_dict(best_model_state)

Epoch 10, Train: 1.0842, Test: 1.0853
Epoch 20, Train: 1.0417, Test: 1.0668
Epoch 30, Train: 0.9956, Test: 1.0512
Epoch 40, Train: 0.9467, Test: 1.0381
Epoch 50, Train: 0.8879, Test: 1.0319

Early stopping at epoch 59


<All keys matched successfully>

In [13]:
torch.save(model.state_dict(), "src/models/movie_model_state.pt")
import pickle
pickle.dump(scaler, open("src/models/scaler.pkl", "wb"))
pickle.dump(movie2idx, open("src/models/movie2idx.pkl", "wb"))

In [14]:
def compute_user_popularity_weight(user_id, df, max_weight=1.0, min_weight=0.0):
    """
    Compute a user-specific popularity weight based on how many movies they have rated.

    - Sparse users → weight closer to max_weight (more popularity-based)
    - Active users → weight closer to min_weight (more model-based)
    """
    n_rated = len(df[df['userId'] == user_id])
    # Example: use a log-scaling for smoother transition
    weight = max_weight * np.exp(-0.1 * n_rated)
    weight = np.clip(weight, min_weight, max_weight)
    return weight

In [15]:
def predict_top_movies(
    user_id, df, model, scaler, genre_cols, movie2idx,
    top_n=10, min_rating=3.5, random_seed=None,
    top_pool_factor=3, noise_std=0.02, popularity_weight=0.2
):
    """
    Predict top N movies for a user with slight randomization and popularity boost.
    Focuses on user-specific features (two_tower_score, user averages) 
    but includes popular movies. Excludes movies the user has already seen.
    """
    import torch
    import numpy as np
    import pandas as pd

    # --- Set seeds for reproducibility if needed ---
    if random_seed is not None:
        np.random.seed(random_seed)
        torch.manual_seed(random_seed)

    # --- Candidate movies (exclude already rated) ---
    rated_movies = df[df['userId'] == user_id]['movieId'].tolist()
    movie_candidates = df[['movieId','title'] + feature_cols].drop_duplicates('movieId')
    movie_candidates = movie_candidates[~movie_candidates['movieId'].isin(rated_movies)]

    if len(movie_candidates) == 0:
        return pd.DataFrame(columns=['title','pred_rating'])

    # --- Feature scaling ---
    X_candidates = movie_candidates[feature_cols].values
    X_scaled = scaler.transform(X_candidates)
    X_tensor = torch.tensor(X_scaled, dtype=torch.float32)

    # --- Embedding indices ---
    movie_idx_tensor = torch.tensor(
        [movie2idx[m] for m in movie_candidates['movieId'].values], dtype=torch.long
    )
    user_idx_tensor = torch.tensor([user_id] * len(movie_candidates), dtype=torch.long)

    # --- Predict using neural network ---
    model.eval()
    with torch.no_grad():
        preds = model(X_tensor, user_idx_tensor, movie_idx_tensor).cpu().numpy().flatten()

    # --- Clip predictions to valid range ---
    preds = np.clip(preds, min_rating, 5)

    # --- Popularity boost ---
    popularity = df.groupby('movieId')['rating'].count()
    popularity = np.log1p(popularity)  # log-scale
    popularity = (popularity - popularity.min()) / (popularity.max() - popularity.min() + 1e-8)
    popularity_scores = movie_candidates['movieId'].map(popularity).fillna(0)

    # --- Combine predictions + popularity + slight noise ---
    combined_scores = preds + popularity_weight * popularity_scores
    combined_scores = np.clip(combined_scores, min_rating, 5)
    combined_scores += np.random.normal(0, noise_std, size=combined_scores.shape)
    combined_scores = np.array(combined_scores)  # force NumPy array

    # --- Build top candidate pool ---
    pool_size = min(len(movie_candidates), top_pool_factor * top_n)
    top_pool_idx = np.argsort(combined_scores)[::-1][:pool_size]
    top_pool = movie_candidates.iloc[top_pool_idx].copy()
    top_pool_scores = combined_scores[top_pool_idx]

    # --- Probabilistic sampling from pool ---
    exp_preds = np.exp(top_pool_scores - top_pool_scores.max())
    probabilities = exp_preds / exp_preds.sum()

    if pool_size > top_n:
        sampled_idx = np.random.choice(pool_size, size=top_n, replace=False, p=probabilities)
        top_pool = top_pool.iloc[sampled_idx].copy()
        top_pool_scores = top_pool_scores[sampled_idx]

    top_pool['pred_rating'] = np.round(top_pool_scores, 2)

    # --- Sort and return ---
    top_movies = top_pool.sort_values('pred_rating', ascending=False)[['title','pred_rating']].reset_index(drop=True)
    top_movies.index += 1
    return top_movies

In [16]:
predict_top_movies(
    user_id=13,
    df=df,
    model=model,
    scaler=scaler,
    genre_cols=genre_cols,
    movie2idx=movie2idx,
    top_n=10
)


,title,pred_rating
1,Pinocchio (1940),4.17
2,Fantasia (1940),4.16
3,"Shawshank Redemption, The (1994)",4.13
4,Bent (1997),4.12
5,Hotel Rwanda (2004),4.09
6,"Usual Suspects, The (1995)",4.06
7,Mephisto (1981),4.06
8,Crimson Tide (1995),4.05
9,Citizen Kane (1941),4.05
10,Dogville (2003),4.05


In [17]:
# Compute binary F1 by thresholding continuous ratings.
# Uses existing `test_preds`, `y_test_tensor`, and `f1_metric` from the notebook.
for thr in (3.0, 3.5):
    f1_metric.reset()
    preds_bin = (test_preds.squeeze() >= thr).int()
    targets_bin = (y_test_tensor.squeeze() >= thr).int()
    f1 = f1_metric(preds_bin, targets_bin)
    print(f"Binary F1 @ threshold {thr}: {f1.item():.4f}")

NameError: name 'f1_metric' is not defined